In [ ]:
import pandas as pd
import numpy as np
import baostock as bs

# Get Stock Data
def get_stock_data(code, start, end, freq='d', adjust='2'):
    fields = 'date,open,high,low,close,volume,amount,turn'

    bs.login()
    
    rs = bs.query_history_k_data_plus(
        code, fields,
        start_date=start, end_date=end,
        frequency=freq, adjustflag=adjust
    )
    
    rows = []
    while (rs.error_code == '0') and rs.next():
        rows.append(rs.get_row_data())
    bs.logout()
    
    if not rows:
        print(f"Warning: no data returned for {code} ({start} to {end})")
        return pd.DataFrame()
    
    df = pd.DataFrame(rows, columns=rs.fields)
    
    # Convert types — baostock returns everything as strings
    df['date'] = pd.to_datetime(df['date'])
    df.set_index('date', inplace=True)
    
    numeric_cols = ['open', 'high', 'low', 'close', 'volume', 'amount', 'turn']
    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
    
    return df


In [ ]:
# Get Stock data for 平安银行
df = get_stock_data('sz.000001', '2024-01-01', '2025-01-01')
df.head()

In [ ]:
# Calculate return manually 
prices = df['close'].values

manual_returns = [np.nan]  # First day has no previous day

for i in range(1, len(prices)):
    prev_price = prices[i-1]
    curr_price = prices[i]
    r = (curr_price - prev_price) / prev_price
    manual_returns.append(r)

df['simple_return_manual'] = manual_returns

df[['close', 'simple_return_manual']].head()

In [ ]:
# Compare manual calculated returns with function call results to verify
df['simple_return'] = df['close'].pct_change()

df[['close', 'simple_return_manual', 'simple_return']].head()

In [ ]:
diff = (df['simple_return_manual'] - df['simple_return']).abs()
print("Max absolute difference:", diff.max())

In [ ]:
i = 10  # pick any row except 0
p_prev = df['close'].iloc[i-1]
p_curr = df['close'].iloc[i]
print(f"Price at row {i-1}: {p_prev}")
print(f"Price at row {i}: {p_curr}")
print(f"Expected:   {(p_curr - p_prev) / p_prev}")
print(f"Manual:     {df['simple_return_manual'].iloc[i]}")
print(f"pct_change: {df['simple_return'].iloc[i]}")

In [ ]:
df['log_return'] = np.log(df['close'] / df['close'].shift(1))

df[['close', 'simple_return', 'log_return']].head(10)

In [ ]:
df[['simple_return', 'log_return']].describe()

In [ ]:
import matplotlib
import matplotlib.pyplot as plt

# Style first, because it resets font-related rcParams
plt.style.use('seaborn-v0_8-whitegrid')


# Then fonts, so they win
matplotlib.rcParams['font.family'] = 'sans-serif'
matplotlib.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei', 'DejaVu Sans']
matplotlib.rcParams['axes.unicode_minus'] = False

df['difference'] = df['simple_return'] - df['log_return']

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(df.index, df['difference'])
ax.set_title(f'Simple Return minus Log Return: {df.index[0].date()} to {df.index[-1].date()}')
ax.set_ylabel('Difference')
ax.axhline(0, color='gray', linewidth=0.5)
plt.tight_layout()
plt.show()

In [ ]:
# Find the day with the largest absolute simple return
max_move_day = df['simple_return'].abs().idxmax()

prev_close = df['close'].shift(1).loc[max_move_day]
this_close = df.loc[max_move_day, 'close']

print(f"Date of biggest move: {max_move_day.date()}")
print(f"Previous close: {prev_close:.2f}")
print(f"That day close: {this_close:.2f}")
print(f"Simple return: {df.loc[max_move_day, 'simple_return']:.6f}")
print(f"Log return:    {df.loc[max_move_day, 'log_return']:.6f}")
print(f"Difference:    {df.loc[max_move_day, 'difference']:.6f}")

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

ax.plot(df.index, df['simple_return'], color='steelblue', linewidth=1)
ax.set_title('SZ..000001 Simple Return')
ax.set_xlabel('日期')
ax.set_ylabel('百分比')
ax.grid(True, alpha=0.3)

plt.show()